# Advanced PyTorch Concepts

In [ ]:

import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader


In [ ]:

transform = transforms.ToTensor()

train_data = datasets.FashionMNIST(".", train=True, download=True, transform=transform)
train_loader = DataLoader(train_data, batch_size=64, shuffle=True)


## Custom Layer

In [ ]:

class GaussianNoise(nn.Module):
    def __init__(self, std):
        super().__init__()
        self.std = std

    def forward(self, x):
        if self.training:
            noise = torch.randn_like(x) * self.std
            return x + noise
        return x


## Custom Model

In [ ]:

class SimpleNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.noise = GaussianNoise(0.1)
        self.fc1 = nn.Linear(784,128)
        self.fc2 = nn.Linear(128,10)

    def forward(self,x):
        x=self.flatten(x)
        x=self.noise(x)
        x=torch.relu(self.fc1(x))
        return self.fc2(x)


## Custom Optimizer and Scheduler

In [ ]:

model=SimpleNet()
optimizer=optim.SGD(model.parameters(),lr=0.01,momentum=0.9)
scheduler=optim.lr_scheduler.StepLR(optimizer,step_size=2,gamma=0.5)
criterion=nn.CrossEntropyLoss()


## Training Loop

In [ ]:

for epoch in range(3):
    print("Epoch",epoch)

    for images,labels in train_loader:

        optimizer.zero_grad()
        outputs=model(images)
        loss=criterion(outputs,labels)

        loss.backward()
        optimizer.step()

    scheduler.step()
    print("loss",loss.item())
